# 01 · Limpeza dos dados

**Projeto Final · Visualização de Dados · ** — Etapa 2: Preparação dos dados

Este notebook prepara os dois datasets do projeto e responde às perguntas da Etapa 1 só com números:

Cada bloco de código vem depois de uma célula de texto que explica **o que** foi feito e **por quê**.

### Perguntas da Etapa 1

**Músicas (Spotify)**
1. Quais gêneros de playlist têm as músicas mais populares?
2. A popularidade das músicas muda conforme o ano de lançamento?
3. Características do áudio (energia, acusticidade, fala, ao vivo) têm relação com a popularidade?
4. As músicas ficaram mais curtas ao longo das décadas?

**Filmes (TMDB)**

5. Quais gêneros de filme têm as maiores notas médias?
6. Como a quantidade de filmes e a nota média mudam por década de lançamento?
7. Filmes com mais votos também têm notas mais altas?
8. Quais idiomas originais aparecem mais e como são avaliados?

In [9]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
from src.utils import DATA, LIMPOS, GENEROS_MUSICA, IDIOMAS

pd.set_option("display.max_columns", 30)
LIMPOS.mkdir(exist_ok=True)

---
## Parte A · Músicas (Spotify)

### A1. Carregar
O dataset `spotify_songs.csv` lista músicas de playlists do Spotify, com popularidade (0 a 100), gênero da playlist e características de áudio calculadas pelo Spotify.

In [ ]:
musicas = pd.read_csv(DATA / "musica" / "spotify_songs.csv")

: 

: 

### A2. Inspecionar
Antes de mexer, conferimos o tamanho, os tipos e os valores ausentes.

In [ ]:
print("Linhas x colunas:", musicas.shape)
musicas.info()

Linhas x colunas: (32833, 23)
<class 'pandas.DataFrame'>
RangeIndex: 32833 entries, 0 to 32832
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  32833 non-null  str    
 1   track_name                32828 non-null  str    
 2   track_artist              32828 non-null  str    
 3   track_popularity          32833 non-null  int64  
 4   track_album_id            32833 non-null  str    
 5   track_album_name          32828 non-null  str    
 6   track_album_release_date  32833 non-null  str    
 7   playlist_name             32833 non-null  str    
 8   playlist_id               32833 non-null  str    
 9   playlist_genre            32833 non-null  str    
 10  playlist_subgenre         32833 non-null  str    
 11  danceability              32833 non-null  float64
 12  energy                    32833 non-null  float64
 13  key                       32833 non-null  

: 

: 

: 

: 

: 

: 

In [ ]:
musicas.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


: 

: 

: 

: 

: 

: 

In [ ]:
musicas.isna().sum()[lambda s: s > 0]

track_name          5
track_artist        5
track_album_name    5
dtype: int64

: 

: 

: 

: 

: 

: 

**O que a inspeção mostrou:**
- 32.833 linhas e 23 colunas.
- A data de lançamento do álbum (`track_album_release_date`) está como **texto** e com formatos diferentes: `2019-06-14`, `2019-06` ou só `2019`.
- A duração está em milissegundos, pouco legível.
- 5 músicas não têm nome, artista nem álbum.
- Uma mesma música (`track_id`) aparece em várias playlists, o que é esperado. Mas também há músicas repetidas **dentro da mesma playlist**, o que é duplicação de verdade (tratada no passo A6).

### A3. Selecionar e renomear
Ficamos só com as colunas que ajudam a responder às perguntas 1 a 4 e traduzimos os nomes para português (seguindo o tratamento que a equipe já tinha começado em `songs_tratamento.ipynb`).

Mantivemos `track_id` e `playlist_id` por enquanto, porque eles são necessários para encontrar duplicados. `playlist_id` é descartado depois do passo A6.

Removemos `danceability`, `key`, `loudness`, `mode`, `instrumentalness`, `valence` e `tempo`, que não entram nas perguntas.

In [ ]:
colunas_descartadas = ["track_album_id", "danceability", "key", "loudness", "mode",
                       "instrumentalness", "valence", "tempo"]
musicas = musicas.drop(columns=colunas_descartadas)

musicas = musicas.rename(columns={
    "track_id": "id_musica",
    "track_name": "nome_musica",
    "track_artist": "artista",
    "track_popularity": "popularidade_musica",
    "track_album_name": "album_musica",
    "track_album_release_date": "data_lancamento_album",
    "playlist_name": "nome_playlist",
    "playlist_id": "id_playlist",
    "playlist_genre": "genero_playlist",
    "playlist_subgenre": "subgenero_playlist",
    "energy": "energia",
    "speechiness": "fala",
    "acousticness": "acusticidade",
    "liveness": "prob_aovivo",
    "duration_ms": "duracao_ms",
})
musicas.columns.tolist()

['id_musica',
 'nome_musica',
 'artista',
 'popularidade_musica',
 'album_musica',
 'data_lancamento_album',
 'nome_playlist',
 'id_playlist',
 'genero_playlist',
 'subgenero_playlist',
 'energia',
 'fala',
 'acusticidade',
 'prob_aovivo',
 'duracao_ms']

: 

: 

: 

: 

: 

: 

### A4. Corrigir tipos
- **Ano de lançamento:** a data vem em três formatos. O ano (4 primeiros caracteres) existe em todos, então extraímos `ano_lancamento` como inteiro e criamos `decada` para a pergunta 4. Não usamos a data completa porque 1.886 linhas só têm o ano.
- **Duração:** convertemos de milissegundos para minutos (`duracao_min`) e **removemos `duracao_ms`**. No notebook anterior, o `drop` não tinha sido atribuído de volta ao DataFrame, então a coluna continuava lá.

In [ ]:
print(musicas["data_lancamento_album"].str.len().value_counts())

musicas["ano_lancamento"] = musicas["data_lancamento_album"].str[:4].astype(int)
musicas["decada"] = (musicas["ano_lancamento"] // 10) * 10

musicas["duracao_min"] = (musicas["duracao_ms"] / 60000).round(2)
musicas = musicas.drop(columns=["duracao_ms", "data_lancamento_album"])

musicas[["ano_lancamento", "decada", "duracao_min"]].describe()

data_lancamento_album
10    30947
4      1855
7        31
Name: count, dtype: int64


,ano_lancamento,decada,duracao_min
count,32833.000000,32833.000000,32833.000000
mean,2011.137270,2004.952639,3.763335
std,11.417451,10.666666,0.997240
min,1957.000000,1950.000000,0.070000
25%,2008.000000,2000.000000,3.130000
50%,2016.000000,2010.000000,3.600000
75%,2019.000000,2010.000000,4.230000
max,2020.000000,2020.000000,8.630000


: 

: 

: 

: 

: 

: 

### A5. Tratar ausentes
Apenas 5 linhas (0,02%) não têm nome, artista e álbum. Sem o nome não dá para identificar a música, e a perda é desprezível, então **removemos** essas linhas.

In [ ]:
print("Antes:", len(musicas))
musicas = musicas.dropna(subset=["nome_musica", "artista"])
print("Depois:", len(musicas))
musicas.isna().sum().sum()

Antes: 32833
Depois: 32828


np.int64(0)

: 

: 

: 

: 

: 

: 

### A6. Remover duplicados
Há dois tipos de repetição, e tratamos cada um de forma diferente:

1. **A mesma música repetida na mesma playlist** (mesmo `id_musica` e `id_playlist`). Isso é erro de coleta e inflaria contagens e médias, então **removemos**.
2. **A mesma música em playlists diferentes.** Isso é informação real (a música está em várias playlists, às vezes de gêneros diferentes), então **mantemos**. Cada linha passa a significar "uma música dentro de uma playlist".

Para as perguntas que são sobre a música em si (2, 3 e 4), usamos a versão **sem repetição de `id_musica`** no passo de estatísticas, para não contar a mesma música duas vezes.

In [ ]:
print("Duplicados exatos:", musicas.duplicated().sum())
print("Mesma música na mesma playlist:", musicas.duplicated(subset=["id_musica", "id_playlist"]).sum())

musicas = musicas.drop_duplicates(subset=["id_musica", "id_playlist"])
musicas = musicas.drop(columns=["id_playlist"])

print("Linhas após remover:", len(musicas))
print("Músicas únicas:", musicas["id_musica"].nunique())

Duplicados exatos: 0
Mesma música na mesma playlist: 582
Linhas após remover: 32246
Músicas únicas: 28352


: 

: 

: 

: 

: 

: 

### A7. Padronizar
- Removemos espaços extras dos textos.
- Trocamos os códigos de gênero (`r&b`, `latin`...) por nomes de exibição (`R&B`, `Latina`...), que vão direto para os gráficos.
- **Durações muito curtas:** há faixas com poucos segundos (vinhetas e intros). Removemos as que têm menos de 1 minuto, porque não são músicas completas e distorcem a pergunta 4.

In [ ]:
for col in ["nome_musica", "artista", "album_musica", "nome_playlist", "subgenero_playlist"]:
    musicas[col] = musicas[col].str.strip()

musicas["genero_playlist"] = musicas["genero_playlist"].map(GENEROS_MUSICA)
print("Gêneros sem correspondência:", musicas["genero_playlist"].isna().sum())

curtas = musicas["duracao_min"] < 1
print("Faixas com menos de 1 minuto:", curtas.sum())
musicas = musicas[~curtas]
print("Linhas finais:", len(musicas))

Gêneros sem correspondência: 0
Faixas com menos de 1 minuto: 25
Linhas finais: 32221


: 

: 

: 

: 

: 

: 

### A8. Salvar

In [ ]:
musicas.to_csv(LIMPOS / "musicas_limpas.csv", index=False)
musicas.head()

,id_musica,nome_musica,artista,popularidade_musica,album_musica,nome_playlist,genero_playlist,subgenero_playlist,energia,fala,acusticidade,prob_aovivo,ano_lancamento,decada,duracao_min
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,I Don't Care (with Justin Bieber) [Loud Luxury...,Pop Remix,Pop,dance pop,0.916,0.0583,0.1020,0.0653,2019,2010,3.25
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,Memories (Dillon Francis Remix),Pop Remix,Pop,dance pop,0.815,0.0373,0.0724,0.3570,2019,2010,2.71
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,All the Time (Don Diablo Remix),Pop Remix,Pop,dance pop,0.931,0.0742,0.0794,0.1100,2019,2010,2.94
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,Call You Mine - The Remixes,Pop Remix,Pop,dance pop,0.930,0.1020,0.0287,0.2040,2019,2010,2.82
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,Someone You Loved (Future Humans Remix),Pop Remix,Pop,dance pop,0.833,0.0359,0.0803,0.0833,2019,2010,3.15


: 

: 

: 

: 

: 

: 